1. Install Required Packages

In [ ]:
import sys
!{sys.executable} -m pip install -r requirements.txt



ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


Get Embed Token & URL

In [ ]:
import os
import requests
from azure.identity import ClientSecretCredential

import json
from flask import Flask, request, jsonify, send_from_directory
from dotenv import load_dotenv
from msal import ConfidentialClientApplication
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

# ---------------------------
# CONFIGURATION - Replace with your values
# ---------------------------

In [2]:
TENANT_ID = os.getenv("AZURE_TENANT_ID", "07897556-e7d2-4d30-8ce7-1947d1d23ad3")
CLIENT_ID = os.getenv("AZURE_CLIENT_ID", "636df4dd-b0e1-4f01-8e43-269a0e2cbac5")
WORKSPACE_ID = "dcae3e53-0f54-4645-9ed2-ba78cfbb0743"
REPORT_ID = "9e84c96e-d01a-49b3-8f62-ec3d85c917ab"
# Azure Power BI API endpoint
POWER_BI_API = "https://api.powerbi.com/v1.0/myorg"
#RLS
DATASET_ID="48b7d02a-9670-4584-bd1c-c8162644f280"
ROLE_NAME="Resellers"
USER_NAME="AW00000221@netway.it"

In [3]:
VAULT_URL = "https://pvlab-ea012e-keyvault.vault.azure.net/"
SECRET_NAME = "AZURE-CLIENT-SECRET"

cred = DefaultAzureCredential()
client = SecretClient(vault_url=VAULT_URL, credential=cred)
secret_value = client.get_secret(SECRET_NAME).value
CLIENT_SECRET = secret_value

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' package to be installed. You

ClientAuthenticationError: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' p

# ---------------------------
# AUTHENTICATION
# ---------------------------

In [25]:
try:
    credential = ClientSecretCredential(
        tenant_id=TENANT_ID,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET
    )

    # Get Azure AD token for Power BI API
    token = credential.get_token("https://analysis.windows.net/powerbi/api/.default")
    access_token = token.token
except Exception as e:
    raise SystemExit(f"Authentication failed: {e}")

In [26]:
print(access_token)

eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIsIng1dCI6InNNMV95QXhWOEdWNHlOLUI2ajJ4em1pazVBbyIsImtpZCI6InNNMV95QXhWOEdWNHlOLUI2ajJ4em1pazVBbyJ9.eyJhdWQiOiJodHRwczovL2FuYWx5c2lzLndpbmRvd3MubmV0L3Bvd2VyYmkvYXBpIiwiaXNzIjoiaHR0cHM6Ly9zdHMud2luZG93cy5uZXQvMDc4OTc1NTYtZTdkMi00ZDMwLThjZTctMTk0N2QxZDIzYWQzLyIsImlhdCI6MTc3MjU2NzAwOCwibmJmIjoxNzcyNTY3MDA4LCJleHAiOjE3NzI1NzA5MDgsImFpbyI6ImsyWmdZRGhsRmRqbnJYRjJidjZUR3RsSGR3UmxBUT09IiwiYXBwaWQiOiI2MzZkZjRkZC1iMGUxLTRmMDEtOGU0My0yNjlhMGUyY2JhYzUiLCJhcHBpZGFjciI6IjEiLCJpZHAiOiJodHRwczovL3N0cy53aW5kb3dzLm5ldC8wNzg5NzU1Ni1lN2QyLTRkMzAtOGNlNy0xOTQ3ZDFkMjNhZDMvIiwiaWR0eXAiOiJhcHAiLCJvaWQiOiIxYmJjNGVmMy1mZWMzLTQ3YzMtOTFkYS0xZmYyMzU3NDdlY2MiLCJyaCI6IjEuQVhNQVZuV0pCOUxuTUUyTTV4bEgwZEk2MHdrQUFBQUFBQUFBd0FBQUFBQUFBQUFBQUFCekFBLiIsInN1YiI6IjFiYmM0ZWYzLWZlYzMtNDdjMy05MWRhLTFmZjIzNTc0N2VjYyIsInRpZCI6IjA3ODk3NTU2LWU3ZDItNGQzMC04Y2U3LTE5NDdkMWQyM2FkMyIsInV0aSI6IlR3aXVPcHFsWDAydERaZnczaHZaQUEiLCJ2ZXIiOiIxLjAiLCJ4bXNfYWN0X2ZjdCI6IjMgOSIsInhtc19mdGQiOiJkR2ZtOGNvM3hDTklDUWdUU0V

# ---------------------------
# GET REPORT DETAILS
# ---------------------------

In [27]:
try:
    headers = {"Authorization": f"Bearer {access_token}"}
    report_url = f"{POWER_BI_API}/groups/{WORKSPACE_ID}/reports/{REPORT_ID}"
    report_response = requests.get(report_url, headers=headers)
    report_response.raise_for_status()
    report_data = report_response.json()

    embed_url = report_data["embedUrl"]
    print(f"Report Name: {report_data['name']}")
    print(f"Embed URL: {embed_url}")
except Exception as e:
    raise SystemExit(f"Failed to get report details: {e}")

Report Name: AdventureWorks Sales
Embed URL: https://app.powerbi.com/reportEmbed?reportId=9e84c96e-d01a-49b3-8f62-ec3d85c917ab&groupId=dcae3e53-0f54-4645-9ed2-ba78cfbb0743&w=2&config=eyJjbHVzdGVyVXJsIjoiaHR0cHM6Ly9XQUJJLUVVUk9QRS1OT1JUSC1CLXJlZGlyZWN0LmFuYWx5c2lzLndpbmRvd3MubmV0IiwiZW1iZWRGZWF0dXJlcyI6eyJ1c2FnZU1ldHJpY3NWTmV4dCI6dHJ1ZX19


# =========== DISCOVERY: datasetId effettivo del report ===========
# Evita mismatch: alcuni errori 400 arrivano perché si passa l'ID del report al posto del dataset,
# oppure perché il report è legato a un dataset diverso da quello atteso.


In [28]:
rep = requests.get(f"{POWER_BI_API}/groups/{WORKSPACE_ID}/reports/{REPORT_ID}", headers=headers)
if not rep.ok:
    req_id = rep.headers.get("RequestId") or rep.headers.get("x-ms-request-id")
    sys.exit(f"[{rep.status_code}] GET report failed. RequestId={req_id}\n{rep.text}")

report_json = rep.json()
dataset_ids = set()

# La maggior parte dei report espone 'datasetId'
if report_json.get("datasetId"):
    dataset_ids.add(report_json["datasetId"])

# Aggiungi/forza anche il DATASET_ID fornito, se vuoi “validarlo” contro il discovery
if DATASET_ID:
    dataset_ids.add(DATASET_ID)

if not dataset_ids:
    sys.exit("Nessun datasetId trovato per il report. Verifica binding/composite models.")
#print(dataset_ids)
# Normalizza ruoli e dataset in liste (lo schema API lo richiede)
roles_list    = [ROLE_NAME] if isinstance(ROLE_NAME, str) else list(ROLE_NAME)
datasets_list = list(dataset_ids)

# ---------------------------
# GENERATE EMBED TOKEN
# ---------------------------

In [29]:
embed_token_url = f"{POWER_BI_API}/GenerateToken"
embed_token_body = {
        "reports": [{"id": REPORT_ID, "groupId": WORKSPACE_ID}],
        "datasets": [{"id": DATASET_ID}],
        "identities": [
        {
            "username": USER_NAME,
            "roles": roles_list,         
            "datasets": datasets_list   
        }
        ]
    }
try:
    embed_token_response = requests.post(embed_token_url, headers=headers, json=embed_token_body)
    if not embed_token_response.ok:
        req_id = embed_token_response.headers.get("RequestId") or embed_token_response.headers.get("x-ms-request-id")
        raise SystemExit(
                    f"[{embed_token_response.status_code}] GenerateToken(v2) failed. RequestId={req_id}\n{embed_token_response.text}"
                )
    #embed_token_response.raise_for_status()
    embed_token = embed_token_response.json()["token"]

    print(f"Embed Token: {embed_token}")
except Exception as e:
    raise SystemExit(f"Failed to generate embed token: {e}")

Embed Token: H4sIAAAAAAAEAB2Txw6sWAJD_-VtaYlUpJZ6Qc45FjsoLjnHgtH8-1TP2rZk-cj_-eNkdz9lxZ-__7yAZj-CvPaITLw5eto-D3EGr6SPUVIDUbYawWoYBZKF4wkGGtBzplzPfE2d4B2teDJhnDVXILjoixPX-NNuRA4n_QXLNv5g0uvdBxmjLs5bBjHE2oCPRWhcscZQvP0SscftVbRXuah12NHzrxvvp8EaOgvW5lGwJihyGZ3yrVoBDRUhr3wBs19bbj4y8sxYNs9u-fqcldyYzrrRSQaF7VaJs0FwrEetTuzVX-MUo7bJIVRV48228LB-Oav6bsQHueU15ZAy23b8ule5hM30CrpZ8Fy56IgvxtmJsdxdrDaOaLWBN-nMXIKyteEME1v5dgVKGIbeBFoVzoelbyskBWHX0AEncNtJEZDHNirOI6iZOzYXUUXqEwXaLUfCVxcyp5PMSN553PvAPJHZhZAa-eXru5m-WC_ne8knFXyt8O14A_mWGTFVhJA4XQTJWLuJLc7tByDIXwT2vota3yos31j2skgyZm7HXBvSUKzbHcvMVI2uscHK1uiAru-adx9Eiqoc0wBUWkgjvrJeSkG7aWu6jYayV_4YXHrmHSfePXZcmwL6Ij5OXPH4CD_Bp8rgxDdq4ULAW9dj2P7Oapa4UI1C1qlRCFhKMHaxPat9JoVZguFYAvpo12euMEZE5rHHYCrYDilWHg100YmLQrON-ZYM79JXdkQ9xsVLSKqn0QqosIPnjHj71WEj67R05ZNsCXR0ZRlPLsjF64bQK-C5HDlxPtuhf6GLskXhbiFsYybukr8FI9S8BLUndAgsYec4iYbOIRZTy94vCnRW5Te_VlEgWUm7N-YSJSZ742psmP3wSPoz7DSZY7p00iPj7R8sw9CyPDrZ0uLQPU6VwL6Jvt16KaCJlD8ijkI1SQGEaEDaT1x60xIztosrgZ_kJEaEiDBeK7lnS3o-afI